In [3]:
from z3 import *


In [11]:
x, y = Ints('x y')
solve(x + 2 * y == 7, x < 2, y < 5)


[y = 3, x = 1]


In [20]:
x = Int('x')
print(type(x))
print(type(x > 3))
print(x > 3)
print((x > 3).sort())
print(RealVal(1)/3, 1/3)

if x > 3:
    print("yes")


<class 'z3.z3.ArithRef'>
<class 'z3.z3.BoolRef'>
x > 3
Bool
1/3 0.3333333333333333


Z3Exception: Symbolic expressions cannot be cast to concrete Boolean values.

In [25]:
print(Int('x') is Int('x'))
print(Int('x').eq(Int('x')))
print(Int('x').eq(Real('x')))

badList = [Int('x') for index in range(3)]
goodList = [Int(f'x{index}') for index in range(3)]
print(type(badList))
solve([variable > 0 for variable in badList] + [badList[0] != badList[1]])


False
True
False
<class 'list'>
no solution


In [43]:
a, b = Ints('a b')
solve(a > 0, b > 0, a*2 + b*2 == 20, a * b == 22)

solver = Solver()
realValue = Real('x')
solver.add(realValue * realValue == 2, realValue > 0)
solver.check()
model = solver.model()
print(model[realValue].sexpr())       # (root-obj (+ (^ x 2) (- 2)) 2)
print(model[realValue].sort())        # Real
print(model[realValue].approx(10))    # a rational approximation


no solution
(root-obj (+ (^ x 2) (- 2)) 2)
Real
388736063997/274877906944


In [37]:
x = Int('x')
solve(x*x == 2)
x = Real('x')
solve(x*x == 2, x > 0)
x = BitVec('x', 8)
solve(x*x == 2)


no solution
[x = 1.4142135623?]
no solution


In [4]:
realValue = Real('x')
solver = Solver()
solver.add(2**realValue == 3)
print(solver.check())
print(solver.reason_unknown())


unknown
unknown


In [40]:
x, y = Ints('x y')
print(simplify(2*x + 2*x))
print(simplify(x + y + 2*x + 3))
print(simplify(x < y + x + 2))


4*x
3 + 3*x + y
Not(y <= -2)


In [44]:
dog, cat, mouse = Ints('dog cat mouse')
solve(dog >= 1, cat >= 1, mouse >= 1,
      dog + cat + mouse == 100,
      1500*dog + 100*cat + 25*mouse == 10000)


[cat = 41, mouse = 56, dog = 3]


In [48]:
print(type(BoolVal(True)))   # BoolRef — a leaf, value fixed
print(type(Bool('p')))       # BoolRef — a leaf, no value
print(type(x > 3))      # BoolRef — a node with two children

expression = x > 3
print(expression.decl().name())   # >
print(expression.children())      # [x, 3]
print(is_const(expression))       # False — it's a node, not a leaf
print(is_true(BoolVal(True)))     # True


<class 'z3.z3.BoolRef'>
<class 'z3.z3.BoolRef'>
<class 'z3.z3.BoolRef'>
>
[x, 3]
False
True


In [5]:
solver = Solver()          # you may have skipped this
solver.add(realValue * realValue == 2, realValue > 0)
print(solver.check())
solver.assertions()


sat


In [18]:
solver = Solver()
x = Int('x')
y = Int('y')
solver.add(x > 10)
solver.add(y == x + 2)
print(solver)
print(solver.assertions())
print(solver.check())
model = solver.model()

for declaration in model.decls():
    print(declaration.name(), '=', model[declaration])

print(model[x].as_long())
print(model[y].as_long())


[x > 10, y == x + 2]
[x > 10, y == x + 2]
sat
y = 13
x = 11
11
13


In [20]:
solver.push()
solver.add(y < 11)
print(solver.check())
print(solver.model())
solver.pop()
print(solver.check())


unsat


Z3Exception: model is not available

In [21]:
solver = Solver()
solver.add(x + y == 10)
solver.add(x > 0)
solver.add(y > 0)
solver.push()
solver.add(x == y)
print(solver.check())
solver.pop()
solver.push()
solver.add(x > 7)
print(solver.check())
solver.pop()
solver.push()
solver.add(x * y == 21)
print(solver.check())
solver.pop()
solver.push()
solver.add(x * y == 30)
print(solver.check())
solver.pop()


sat
sat
sat
unsat


In [30]:
intValue = Int("i")
realValue = Real("r")
boolValue = Bool("p")
bitValue = BitVec("b", 32)

solver = Solver()
solver.add(intValue == 2)
solver.add(realValue == RealVal("1/3"))
solver.add(boolValue == True)
solver.add(bitValue == 0xDEADBEEF)
solver.check()
print(solver.model())

model = solver.model()

print(model[intValue].as_long())
print(model[realValue])


[b = 3735928559, p = True, i = 2, r = 1/3]
2
1/3


In [35]:
z = Real('z')
solver = Solver()
solver.add(z**2 == 2)
solver.check()
model = solver.model()
print(model[z])
print(model[z].sexpr())
print(model[z].approx(10).denominator_as_long())


-1.4142135623?
(root-obj (+ (^ x 2) (- 2)) 1)
2199023255552


In [40]:
xValue, yValue = Ints('x y')

solver = Solver()
solver.add(xValue > 5)          # y is declared in Python but never constrained
print(solver.check())

model = solver.model()
print(model)                              # what's in it?
print(model.decls())                      # which symbols does it interpret?
print(model[yValue])                      # ?
print(model.eval(yValue, model_completion=True))


sat
[x = 6]
[x]
None
0


In [43]:
a, b = Ints('a b')

solver = Solver()
solver.add(1 <= a)
solver.add(b <= 4)

while solver.check() == sat:
    model = solver.model()
    print(model)
    solver.add(a != model[a])
    solver.add(b != model[b])


[a = 1, b = 4]
[a = 2, b = 0]
[b = -1, a = 3]
[a = 4, b = -2]
[a = 5, b = 1]
[a = 6, b = 2]
[b = 3, a = 7]
[a = 8, b = -3]
[b = -4, a = 9]
[a = 10, b = -5]
[b = -6, a = 11]
[a = 12, b = -7]
[b = -8, a = 13]
[a = 14, b = -9]
[b = -10, a = 15]
[b = -11, a = 16]
[a = 17, b = -12]
[b = -13, a = 18]
[b = -14, a = 19]
[b = -15, a = 20]
[b = -16, a = 21]
[b = -17, a = 22]
[b = -18, a = 23]
[b = -19, a = 24]
[b = -20, a = 25]
[b = -21, a = 26]
[a = 27, b = -22]
[a = 28, b = -23]
[a = 29, b = -24]
[b = -25, a = 30]
[b = -26, a = 31]
[b = -27, a = 32]
[b = -28, a = 33]
[a = 34, b = -29]
[b = -30, a = 35]
[a = 36, b = -31]
[a = 37, b = -32]
[a = 38, b = -33]
[a = 39, b = -34]
[a = 40, b = -35]
[b = -36, a = 41]
[a = 42, b = -37]
[a = 43, b = -38]
[b = -39, a = 44]
[b = -40, a = 45]
[b = -41, a = 46]
[a = 47, b = -42]
[a = 48, b = -43]
[a = 49, b = -44]
[a = 50, b = -45]
[a = 51, b = -46]
[b = -47, a = 52]
[b = -48, a = 53]
[a = 54, b = -49]
[b = -50, a = 55]
[a = 56, b = -51]
[a = 57, b = -52]
[b

[a = 224, b = -219]
[a = 225, b = -220]
[b = -221, a = 226]
[a = 227, b = -222]
[a = 228, b = -223]
[b = -224, a = 229]
[a = 230, b = -225]
[b = -226, a = 231]
[b = -227, a = 232]
[b = -228, a = 233]
[a = 234, b = -229]
[a = 235, b = -230]
[b = -231, a = 236]
[b = -232, a = 237]
[a = 238, b = -233]
[b = -234, a = 239]
[a = 240, b = -235]
[b = -236, a = 241]
[b = -237, a = 242]
[a = 243, b = -238]
[b = -239, a = 244]
[a = 245, b = -240]
[b = -241, a = 246]
[b = -242, a = 247]
[b = -243, a = 248]
[b = -244, a = 249]
[b = -245, a = 250]
[b = -246, a = 251]
[b = -247, a = 252]
[b = -248, a = 253]
[a = 254, b = -249]
[a = 255, b = -250]
[a = 256, b = -251]
[a = 257, b = -252]
[b = -253, a = 258]
[b = -254, a = 259]
[b = -255, a = 260]
[a = 261, b = -256]
[b = -257, a = 262]
[b = -258, a = 263]
[a = 264, b = -259]
[b = -260, a = 265]
[b = -261, a = 266]
[a = 267, b = -262]
[a = 268, b = -263]
[b = -264, a = 269]
[a = 270, b = -265]
[a = 271, b = -266]
[a = 272, b = -267]
[b = -268, a = 273]


[a = 351, b = -346]
[a = 352, b = -347]
[b = -348, a = 353]
[b = -349, a = 354]
[a = 355, b = -350]
[b = -351, a = 356]
[b = -352, a = 357]
[b = -353, a = 358]
[b = -354, a = 359]
[a = 360, b = -355]
[a = 361, b = -356]
[a = 362, b = -357]
[a = 363, b = -358]
[a = 364, b = -359]
[b = -360, a = 365]
[a = 366, b = -361]
[a = 367, b = -362]
[b = -363, a = 368]
[b = -364, a = 369]
[b = -365, a = 370]
[b = -366, a = 371]
[b = -367, a = 372]
[b = -368, a = 373]
[a = 374, b = -369]
[a = 375, b = -370]
[b = -371, a = 376]
[a = 377, b = -372]
[a = 378, b = -373]
[b = -374, a = 379]
[b = -375, a = 380]
[b = -376, a = 381]
[b = -377, a = 382]
[b = -378, a = 383]
[a = 384, b = -379]
[a = 385, b = -380]
[b = -381, a = 386]
[b = -382, a = 387]
[a = 388, b = -383]
[a = 389, b = -384]
[b = -385, a = 390]
[b = -386, a = 391]
[b = -387, a = 392]
[b = -388, a = 393]
[a = 394, b = -389]
[b = -390, a = 395]
[b = -391, a = 396]
[b = -392, a = 397]
[b = -393, a = 398]
[b = -394, a = 399]
[b = -395, a = 400]


[a = 447, b = -442]
[b = -443, a = 448]
[a = 449, b = -444]
[b = -445, a = 450]
[b = -446, a = 451]
[a = 452, b = -447]
[b = -448, a = 453]
[b = -449, a = 454]
[b = -450, a = 455]
[b = -451, a = 456]
[b = -452, a = 457]
[a = 458, b = -453]
[a = 459, b = -454]
[a = 460, b = -455]
[b = -456, a = 461]
[b = -457, a = 462]
[b = -458, a = 463]
[a = 464, b = -459]
[a = 465, b = -460]
[a = 466, b = -461]
[b = -462, a = 467]
[a = 468, b = -463]
[a = 469, b = -464]
[a = 470, b = -465]
[b = -466, a = 471]
[a = 472, b = -467]
[b = -468, a = 473]
[b = -469, a = 474]
[b = -470, a = 475]
[b = -471, a = 476]
[b = -472, a = 477]
[b = -473, a = 478]
[b = -474, a = 479]
[a = 480, b = -475]
[b = -476, a = 481]
[b = -477, a = 482]
[a = 483, b = -478]
[b = -479, a = 484]
[b = -480, a = 485]
[b = -481, a = 486]
[b = -482, a = 487]
[a = 488, b = -483]
[a = 489, b = -484]
[a = 490, b = -485]
[b = -486, a = 491]
[a = 492, b = -487]
[b = -488, a = 493]
[a = 494, b = -489]
[a = 495, b = -490]
[a = 496, b = -491]


[a = 523, b = -518]
[a = 524, b = -519]
[b = -520, a = 525]
[a = 526, b = -521]
[a = 527, b = -522]
[b = -523, a = 528]
[a = 529, b = -524]
[a = 530, b = -525]
[a = 531, b = -526]
[b = -527, a = 532]
[b = -528, a = 533]
[a = 534, b = -529]
[a = 535, b = -530]
[b = -531, a = 536]
[b = -532, a = 537]
[a = 538, b = -533]
[b = -534, a = 539]
[a = 540, b = -535]
[b = -536, a = 541]
[b = -537, a = 542]
[b = -538, a = 543]
[b = -539, a = 544]
[b = -540, a = 545]
[b = -541, a = 546]
[b = -542, a = 547]
[b = -543, a = 548]
[b = -544, a = 549]
[a = 550, b = -545]
[b = -546, a = 551]
[b = -547, a = 552]
[a = 553, b = -548]
[a = 554, b = -549]
[b = -550, a = 555]
[a = 556, b = -551]
[b = -552, a = 557]
[a = 558, b = -553]
[a = 559, b = -554]
[a = 560, b = -555]
[b = -556, a = 561]
[b = -557, a = 562]
[b = -558, a = 563]
[b = -559, a = 564]
[b = -560, a = 565]
[b = -561, a = 566]
[a = 567, b = -562]
[b = -563, a = 568]
[a = 569, b = -564]
[a = 570, b = -565]
[b = -566, a = 571]
[a = 572, b = -567]


...

[b = -3181, a = 3186]
[b = -3182, a = 3187]


[b = -3183, a = 3188]
[a = 3189, b = -3184]


[b = -3185, a = 3190]
[a = 3191, b = -3186]
[b = -3187, a = 3192]
[b = -3188, a = 3193]
[b = -3189, a = 3194]
[a = 3195, b = -3190]


[b = -3191, a = 3196]
[a = 3197, b = -3192]


[b = -3193, a = 3198]
[b = -3194, a = 3199]
[b = -3195, a = 3200]
